In [1]:
import torch
import importlib
import numpy as np
from scipy.optimize import minimize
from modele import *
from visualisation import *
import pytorch_optim
importlib.reload(pytorch_optim)
from pytorch_optim import *
%load_ext autoreload
%autoreload 2


In [2]:
N=10
nb_colonnes_default = 50
T = 300
L = 5
P0 = torch.tensor([[0, 1 / 2, 1], [1 / 2, 0, 0], [1 / 2, 1 / 2, 0]]) 
params = {
    "N": N,  # nombre de couches
    "L": L,  # longueur du bassin
    "H": 1,  # hauteur du bassin
    "nb_colonnes_default": 50,  # discrétisation horizontale
    "D": 1e-5,  # coefficient de diffusion
    "T": T,  # temps final pour 20 tours
    "CFL": 0.95,  # facteur CFL
    "u" : [0.1]*N,
    "I_s": 2000, #1500 initialement
    "epsilon": 3, #4.6 pour une luminosité de 1% au fond du bassin
    "k_d": 2.99e-4,   # 2.99*10e-5 = 2.99e-4
    "k_r": 4.8e-4,    # 4.8*10e-5 = 4.8e-4
    "k_h": 3.64e-4,   # 3.64*10e-5 = 3.64e-4
    "tau": 6.849,
    "sigma_H": 2.9e-3,  # 2.9*10e-4 = 2.9e-3
    "theta": 3.64e-4 * 2.9e-3,  # k_h * sigma_H
    "I_star": np.sqrt(4.8e-4 / (2.99e-4 * 6.849 * (2.9e-3)**2)),
    "mu_max": 3.64e-4 * 2.9e-3 / (6.849 * 2.9e-3 + 2*np.sqrt((2.99e-4 * 6.849 * (2.9e-3)**2)/4.8e-4))
}

In [10]:

P, biomass_list = solve_bistochastique(X_ini_one_layer_torch(N, nb_colonnes_default), T, params, 30, 1000, 0.2, True, torch.eye(N))
print(P)

obj: tensor(50.0076, grad_fn=<AddBackward0>) pen: tensor(0., grad_fn=<SumBackward0>)
obj: tensor(50.0080, grad_fn=<AddBackward0>) pen: tensor(0.3318, grad_fn=<SumBackward0>)
obj: tensor(50.0077, grad_fn=<AddBackward0>) pen: tensor(0.0233, grad_fn=<SumBackward0>)
obj: tensor(50.0074, grad_fn=<AddBackward0>) pen: tensor(0.0861, grad_fn=<SumBackward0>)
obj: tensor(50.0073, grad_fn=<AddBackward0>) pen: tensor(0.1936, grad_fn=<SumBackward0>)
obj: tensor(50.0074, grad_fn=<AddBackward0>) pen: tensor(0.1129, grad_fn=<SumBackward0>)
obj: tensor(50.0073, grad_fn=<AddBackward0>) pen: tensor(0.0151, grad_fn=<SumBackward0>)
obj: tensor(50.0078, grad_fn=<AddBackward0>) pen: tensor(0.0102, grad_fn=<SumBackward0>)
obj: tensor(50.0077, grad_fn=<AddBackward0>) pen: tensor(0.0661, grad_fn=<SumBackward0>)
obj: tensor(50.0076, grad_fn=<AddBackward0>) pen: tensor(0.0963, grad_fn=<SumBackward0>)
obj: tensor(50.0077, grad_fn=<AddBackward0>) pen: tensor(0.0721, grad_fn=<SumBackward0>)
obj: tensor(50.0077, grad

In [ ]:
matrices = {}
biomasses = {}
for l in [1, 5, 10, 25, 50] :
    params["nb_colonnes_default"] = l*10
    params["L"] = l
    P, biomass_list = solve_bistochastique(X_ini_one_layer_torch(N, params["nb_colonnes_default"]), T, params)
    matrices[f'P{l}'] = P
    biomasses[f'B{l}'] = biomass_list
    print(P, biomass_list[-1])

obj: tensor(10.0724, grad_fn=<AddBackward0>) pen: tensor(1.2710, grad_fn=<SumBackward0>)
obj: tensor(10.0720, grad_fn=<AddBackward0>) pen: tensor(0.4676, grad_fn=<SumBackward0>)


Avec Newton :

In [8]:
print(params)

{'N': 3, 'L': 5, 'H': 1, 'nb_colonnes_default': 50, 'D': 1e-05, 'T': 300, 'CFL': 0.95, 'u': [0.01, 0.01, 0.01], 'I_s': 1500, 'epsilon': 4.6, 'k_d': 0.000299, 'k_r': 0.00048, 'k_h': 0.000364, 'tau': 6.849, 'sigma_H': 0.0029, 'theta': 1.0556e-06, 'I_star': np.float64(166.94501039780602), 'mu_max': np.float64(3.315108749636504e-05)}


In [ ]:
N_lignes = params["N"]
Q = torch.nn.Parameter(torch.randn(N_lignes, N_lignes))
def J(Q):
    P = torch.softmax(Q, dim=0)
    return(fonction_objectif_torch(X_ini_one_layer_torch(N_lignes, nb_colonnes_default), T, P, params))

Q_sol, masse_tot = Newton(J, Q, N_lignes**2)
print(torch.softmax(Q_sol, dim=0))
print(f'masse tot : {masse_tot}')

In [4]:
params["N"] = 10
params["u"] = [0.01]*10
N_lignes = 10
Q = torch.nn.Parameter(torch.randn(N_lignes, N_lignes))
def J(Q):
    P = torch.softmax(Q, dim=0)
    return(fonction_objectif_torch(X_ini_one_layer_torch(N_lignes, nb_colonnes_default), T, P, params))

Q_sol = Newton(J, Q, N_lignes**2)
print(torch.softmax(Q_sol, dim=0))

tensor([[1.3290e-14, 2.7162e-02, 0.0000e+00, 1.0938e-01, 2.4756e-01, 1.6755e-01,
         3.6056e-02, 1.6333e-01, 7.8205e-02, 1.5063e-02],
        [2.3543e-14, 7.1905e-02, 0.0000e+00, 6.4791e-03, 5.5823e-02, 1.1783e-01,
         9.0548e-02, 1.4811e-01, 4.8556e-02, 2.7833e-01],
        [1.0000e+00, 1.7997e-02, 1.0000e+00, 4.5417e-02, 7.9803e-02, 7.1097e-02,
         8.4329e-02, 9.3214e-03, 1.8424e-01, 4.3045e-02],
        [1.3179e-14, 1.3997e-01, 0.0000e+00, 1.4435e-01, 3.8766e-02, 2.3223e-02,
         2.8493e-02, 4.0821e-01, 5.8166e-02, 1.1596e-01],
        [3.5437e-14, 4.2584e-03, 0.0000e+00, 4.0058e-02, 6.6528e-02, 1.5439e-01,
         5.8696e-02, 1.6503e-02, 5.0319e-02, 1.5443e-01],
        [1.7814e-14, 1.8955e-01, 0.0000e+00, 2.5302e-02, 1.3777e-01, 2.3559e-01,
         1.1356e-02, 7.7366e-02, 3.6857e-01, 4.9532e-02],
        [2.5497e-14, 2.0290e-02, 0.0000e+00, 7.5081e-02, 1.5246e-01, 1.5904e-01,
         1.0986e-01, 3.2093e-02, 3.8182e-02, 8.3069e-02],
        [1.2876e-13, 1.0745